# streaming-asr1 on Colab GPU + ngrok

Colab GPU에서 STT 서버(FastAPI + NeMo-Speech.cpp CUDA)를 띄우고, ngrok으로 외부에서 접속할 수 있게 한다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cheetah31/streaming-asr/blob/env/colab/colab/streaming_asr1_colab.ipynb)

**사전 준비**
1. 메뉴 `런타임 > 런타임 유형 변경` → **T4 GPU** (또는 다른 GPU)
2. 왼쪽 🔑 **보안 비밀(Secrets)** 에 등록하고 *노트북 액세스* 를 켠다
   | 이름 | 필수 | 설명 |
   |---|---|---|
   | `NGROK_AUTHTOKEN` | ✅ | https://dashboard.ngrok.com/get-started/your-authtoken |
   | `NGROK_DOMAIN` | | ngrok 고정 도메인 (예: `xxx.ngrok-free.app`). 없으면 실행할 때마다 주소가 바뀜 |

위에서부터 순서대로 실행한다.

In [ ]:
REPO_URL = "https://github.com/cheetah31/streaming-asr.git"  # 공개 저장소
BRANCH = "env/colab"
WORKDIR = "/content/streaming-asr1"
PORT = 8100

In [ ]:
!nvidia-smi

## 1. 코드 받기

In [ ]:
import os
import subprocess

from google.colab import userdata


def secret(name):
    try:
        return userdata.get(name)
    except Exception:  # 등록 안 됨 / 노트북 액세스 미허용
        return None


# vendor/ 의 Windows 런타임 DLL(Git LFS)은 Colab 에서 쓰지 않으므로 받지 않는다 (LFS 전송량 절약)
git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if os.path.isdir(f"{WORKDIR}/.git"):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin", BRANCH], check=True, env=git_env)
    subprocess.run(["git", "-C", WORKDIR, "checkout", "-q", BRANCH], check=True, env=git_env)
    subprocess.run(["git", "-C", WORKDIR, "merge", "--ff-only", f"origin/{BRANCH}"], check=True, env=git_env)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, WORKDIR], check=True, env=git_env)
os.chdir(WORKDIR)
!git log --oneline -3

## 2. 환경 준비

uv / Python 의존성, NeMo-Speech.cpp Linux CUDA 런타임(약 100MB), 모델(약 700MB)을 받고 `.env` 를 만든다.
런타임이 유지되는 동안은 다시 실행해도 이미 받은 파일은 건너뛴다.

In [ ]:
!bash scripts/colab_setup.sh

## 3. 서버 + ngrok 터널 시작

`uv run streaming-asr1 --ngrok` 을 백그라운드로 실행한다. 로그는 `server.log`.
ngrok 무료 플랜은 터널(에이전트 세션)을 동시에 하나만 열 수 있다. 다른 곳에서 열려 있으면 실패한다.

In [ ]:
import re
import shutil
import time
import urllib.request

env = dict(os.environ)
env["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{env['PATH']}"
env["NGROK_AUTHTOKEN"] = secret("NGROK_AUTHTOKEN") or ""
if domain := secret("NGROK_DOMAIN"):
    env["NGROK_DOMAIN"] = domain
assert env["NGROK_AUTHTOKEN"], "보안 비밀에 NGROK_AUTHTOKEN 을 등록하고 노트북 액세스를 허용하세요"

if "server" in globals() and server.poll() is None:  # 다시 실행하면 기존 서버를 내린다
    server.terminate()
    server.wait()

log_file = open("server.log", "w")
server = subprocess.Popen(
    [shutil.which("uv", path=env["PATH"]), "run", "streaming-asr1", "--port", str(PORT), "--ngrok"],
    cwd=WORKDIR, env=env, stdout=log_file, stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(300):
    if server.poll() is not None:
        break
    if m := re.search(r"ngrok public URL: (\S+)", open("server.log").read()):
        public_url = m.group(1)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
        if public_url:
            break
    except OSError:
        pass
    time.sleep(1)

print(open("server.log").read()[-3000:])
if server.poll() is not None or not public_url:
    raise RuntimeError("서버 시작 실패 - 위 로그를 확인하세요")
print(f"\n웹 클라이언트 : {public_url}\nSwagger      : {public_url}/docs")

## 4. 외부에서 테스트

- **브라우저 (마이크)**: 위에 출력된 `웹 클라이언트` 주소 → ngrok 안내 페이지에서 **Visit Site** → **녹음 시작**.
  ngrok 이 HTTPS 를 제공하므로 다른 PC/모바일에서도 마이크를 쓸 수 있다.
- **Swagger**: `<public_url>/docs`
- **curl** (API 호출에는 ngrok 안내 페이지가 끼지 않도록 헤더를 붙인다)
  ```
  curl -H "ngrok-skip-browser-warning: 1" <public_url>/health
  curl -H "ngrok-skip-browser-warning: 1" <public_url>/v1/audio/transcriptions -F file=@test.wav -F language=ko
  curl -H "ngrok-skip-browser-warning: 1" <public_url>/api/transcribe/conversation -F file=@meeting.wav -F language=ko
  ```
- **WebSocket**: `wss://<public_url 의 호스트>/ws/transcribe?language=ko-KR`

아래 셀은 ngrok 주소를 거쳐 `/health` 를 호출한다.

In [ ]:
import json

import requests

H = {"ngrok-skip-browser-warning": "1"}
print(json.dumps(requests.get(f"{public_url}/health", headers=H, timeout=10).json(), ensure_ascii=False, indent=2))

WAV 파일을 올려서 ngrok 주소로 대화 음성 테스트(`/api/transcribe/conversation`)를 한다.

In [ ]:
from google.colab import files

for name in files.upload():
    with open(name, "rb") as f:
        r = requests.post(f"{public_url}/api/transcribe/conversation", headers=H,
                          files={"file": f}, data={"language": "ko"}, timeout=600)
    r.raise_for_status()
    res = r.json()
    print(f"[{name}] {res['duration']}s, 처리 {res['elapsed']}s, RTF {res['rtf']}")
    for u in res["utterances"]:
        print(f"  {u['start']:>7}~{u['end']:<7} {u['text']}")
    print(json.dumps(res["stt_ms"], ensure_ascii=False, indent=2))

## 5. 로그 / 종료

In [ ]:
!tail -n 50 server.log

In [ ]:
server.terminate()
server.wait()
print("서버와 ngrok 터널을 종료했습니다")